# 6장 1강: CV 기반 하이퍼파라미터 튜닝과 모델 탐색

## 실습 목표

이번 실습에서는 **Ames Housing** 데이터를 이용하여 하이퍼파라미터 탐색과 교차검증을 결합해 최적 모델을 찾습니다.

- GridSearchCV와 RandomizedSearchCV의 탐색 방식과 비용 차이를 비교합니다.
- Optuna의 TPE 탐색을 이용해 초기 탐색 이후 이전 trial 결과를 활용하는 탐색을 수행합니다.
- 교차검증 평균 점수와 표준편차를 함께 확인합니다.
- 탐색 시간까지 고려하여 최종 후보를 선정합니다.
- 처음부터 분리해 둔 hold-out test set은 최종 모델을 정한 뒤 한 번만 사용합니다.
- 최종 모델과 메타데이터를 저장합니다.

> 이 실습에서는 6장 1강 교안에서 다룬 Grid Search, Random Search, Optuna, CV 평균·표준편차, 탐색 비용, hold-out test, 모델 저장 내용만 사용합니다.


## 실습 환경 / 데이터

- Python
- pandas
- scikit-learn
- scipy
- Optuna
- joblib
- 데이터: `data/Ames_Housing.csv`

### 분석 목표

주택 특성을 이용하여 `SalePrice`를 예측하는 회귀 문제입니다.

### 사용할 특성

- `OverallQual`
- `GrLivArea`
- `GarageCars`
- `GarageArea`
- `TotalBsmtSF`
- `1stFlrSF`
- `2ndFlrSF`
- `FullBath`
- `TotRmsAbvGrd`
- `YearBuilt`
- `YearRemodAdd`
- `Fireplaces`

예측 대상은 `SalePrice`입니다.

## 실습 준비

하이퍼파라미터 탐색에 Test 데이터를 사용하면 안 됩니다.

따라서 먼저 전체 데이터를 **Train / hold-out Test**로 분리하고,

- 하이퍼파라미터 탐색과 교차검증 → Train 데이터만 사용
- 최종 성능 확인 → 모든 선택이 끝난 뒤 hold-out Test를 한 번만 사용

하는 흐름으로 진행합니다.

In [ ]:
import os
import json
import time
import joblib
import optuna
import pandas as pd

from datetime import datetime
from scipy.stats import randint

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score
)
from sklearn.ensemble import RandomForestRegressor

house = pd.read_csv("data/Ames_Housing.csv")

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "FullBath",
    "TotRmsAbvGrd",
    "YearBuilt",
    "YearRemodAdd",
    "Fireplaces"
]

X = house[features]
y = house["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train 크기:", X_train.shape)
print("Hold-out Test 크기:", X_test.shape)


---

# 필수 1. Grid Search와 Random Search 비교

## 배경

GridSearchCV는 지정한 모든 조합을 빠짐없이 평가하고, RandomizedSearchCV는 정해진 횟수만큼 조합을 무작위로 선택합니다.

이번 실습에서는 같은 RandomForestRegressor를 대상으로 두 탐색 전략을 비교합니다.

## 문제 1. GridSearchCV를 수행하세요.

### 요구사항

1. `RandomForestRegressor(random_state=42, n_jobs=-1)`를 사용합니다.
2. 다음 후보를 사용합니다.

```python
grid_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 8, 12]
}
```

3. `cv=5`를 사용합니다.
4. `scoring="r2"`를 사용합니다.
5. 탐색 시간을 측정합니다.
6. `best_params_`와 `best_score_`를 출력합니다.
7. `cv_results_`에서 최적 조합의 `mean_test_score`, `std_test_score`를 확인합니다.

### 확인 포인트

- 총 9개 조합을 모두 평가하는가?
- 5-Fold이므로 각 조합이 5번 평가되는 구조를 이해했는가?
- 평균 점수뿐 아니라 표준편차도 확인했는가?

In [ ]:
# TODO
# GridSearchCV를 수행하세요.
grid_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 8, 12],
}
rf = RandomForestRegressor(random_state=42, n_jobs=-1)
grid = GridSearchCV(
    rf,
    param_grid=grid_params,
    cv=5,
    scoring="r2",
    n_jobs=-1,
)
t0 = time.time()
grid.fit(X_train, y_train)
grid_seconds = time.time() - t0

print("best_params_:", grid.best_params_)
print("best_score_:", grid.best_score_)
best_idx = grid.best_index_
print("mean_test_score:", grid.cv_results_["mean_test_score"][best_idx])
print("std_test_score:", grid.cv_results_["std_test_score"][best_idx])
print("search_seconds:", grid_seconds)

grid_cv_mean = grid.cv_results_["mean_test_score"][best_idx]
grid_cv_std = grid.cv_results_["std_test_score"][best_idx]


### Q1. Grid Search의 가장 큰 장점과 한계는 무엇인가요?

**답변:**  
장점: 지정한 모든 조합을 빠짐없이 평가. 한계: 탐색 공간이 커지면 조합 수·시간이 급증합니다.


## 문제 2. RandomizedSearchCV를 수행하세요.

### 요구사항

1. 같은 RandomForestRegressor를 사용합니다.
2. 다음 탐색 공간을 사용합니다.

```python
random_params = {
    "n_estimators": randint(100, 301),
    "max_depth": randint(4, 13)
}
```

3. `n_iter=9`를 사용합니다.
4. `cv=5`, `scoring="r2"`를 사용합니다.
5. `random_state=42`를 사용합니다.
6. 탐색 시간을 측정합니다.
7. 최적 파라미터, CV 평균 점수, 표준편차를 확인합니다.

### 확인 포인트

- 전체 조합을 모두 보는 것이 아니라 9번만 무작위로 탐색하는가?
- `n_iter`가 탐색 예산을 직접 정한다는 점을 이해했는가?

In [ ]:
# TODO
# RandomizedSearchCV를 수행하세요.
random_params = {
    "n_estimators": randint(100, 301),
    "max_depth": randint(4, 13),
}
rf = RandomForestRegressor(random_state=42, n_jobs=-1)
rand = RandomizedSearchCV(
    rf,
    param_distributions=random_params,
    n_iter=9,
    cv=5,
    scoring="r2",
    random_state=42,
    n_jobs=-1,
)
t0 = time.time()
rand.fit(X_train, y_train)
rand_seconds = time.time() - t0

print("best_params_:", rand.best_params_)
print("best_score_:", rand.best_score_)
best_idx = rand.best_index_
print("mean_test_score:", rand.cv_results_["mean_test_score"][best_idx])
print("std_test_score:", rand.cv_results_["std_test_score"][best_idx])
print("search_seconds:", rand_seconds)

rand_cv_mean = rand.cv_results_["mean_test_score"][best_idx]
rand_cv_std = rand.cv_results_["std_test_score"][best_idx]


### Q2. Random Search가 Grid Search보다 유리할 수 있는 상황은 언제인가요?

**답변:**  
탐색 공간이 넓고 예산(시간/횟수)이 제한될 때, 중요한 축을 더 다양하게 샘플링할 수 있어 유리할 수 있습니다.


---

# 필수 2. Optuna와 CV 결합

## 배경

Grid와 Random은 이전 탐색 결과와 상관없이 후보를 선택합니다.

이번 실습에서는 Optuna의 `TPESampler`를 사용합니다.  
처음 3개의 trial은 초기 탐색으로 수행하고, 이후 trial부터는 앞선 결과를 참고하여 다음 후보를 제안하도록 설정합니다.

## 문제 1. Optuna objective에 교차검증을 연결하세요.

### 요구사항

1. `objective(trial)` 함수를 만듭니다.
2. 다음 하이퍼파라미터를 탐색합니다.
   - `n_estimators`: 100 ~ 300
   - `max_depth`: 4 ~ 12
3. `RandomForestRegressor`를 사용합니다.
4. `cross_val_score(..., cv=5, scoring="r2")`를 사용합니다.
5. CV 평균 점수를 반환합니다.
6. 다음 설정의 `TPESampler`를 만듭니다.

```python
sampler = optuna.samplers.TPESampler(
    seed=42,
    n_startup_trials=3
)
```

7. `direction="maximize"`와 위 sampler를 사용해 study를 만듭니다.
8. `n_trials=9`로 탐색합니다.
9. 탐색 시간을 측정합니다.
10. `best_params`, `best_value`를 출력합니다.

### 확인 포인트

- 한 번의 시도를 trial이라고 부르는가?
- 탐색 전체를 study라고 부르는가?
- 처음 3개 trial 이후에는 이전 trial 결과를 활용하는 TPE 탐색이 이루어지는가?
- CV 평균 성능을 Optuna의 최적화 대상으로 사용했는가?


In [ ]:
# TODO
# Optuna objective + CV를 구성하세요.
# TPESampler(seed=42, n_startup_trials=3)를 사용하고
# 총 9 trial을 실행하세요.

def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 100, 300)
    max_depth = trial.suggest_int("max_depth", 4, 12)
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42,
        n_jobs=-1,
    )
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="r2")
    return scores.mean()

sampler = optuna.samplers.TPESampler(seed=42, n_startup_trials=3)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize", sampler=sampler)

t0 = time.time()
study.optimize(objective, n_trials=9)
optuna_seconds = time.time() - t0

print("best_params:", study.best_params)
print("best_value:", study.best_value)
print("search_seconds:", optuna_seconds)


### Q3. Optuna가 Grid / Random과 다른 핵심은 무엇인가요?

**답변:**  
이전 trial 결과를 활용해 다음 후보를 제안하는 적응적 탐색(TPE 등)을 한다는 점이 핵심입니다.


## 문제 2. Optuna 최적 조합의 CV 표준편차를 확인하세요.

### 요구사항

1. `study.best_params`로 RandomForestRegressor를 만듭니다.
2. 다시 5-Fold `cross_val_score()`를 수행합니다.
3. CV 평균과 표준편차를 계산합니다.

### 확인 포인트

- 최종 후보를 볼 때 평균만 보지 않고 Fold 간 흔들림도 확인했는가?

In [ ]:
# TODO
# Optuna 최적 조합의 CV 평균과 표준편차를 확인하세요.
best_rf = RandomForestRegressor(
    **study.best_params,
    random_state=42,
    n_jobs=-1,
)
optuna_scores = cross_val_score(best_rf, X_train, y_train, cv=5, scoring="r2")
optuna_cv_mean = optuna_scores.mean()
optuna_cv_std = optuna_scores.std()
print("CV mean:", optuna_cv_mean)
print("CV std:", optuna_cv_std)


---

# 과제 1. 최종 모델 선정과 hold-out Test 확인

## 배경

세 탐색 전략에서 각각 최적 후보를 얻었습니다.

최종 모델은 **CV 평균을 우선 기준으로 비교**하되, 성능 차이가 매우 작다면 다음 항목도 함께 확인합니다.

- CV 표준편차
- 탐색 시간

그리고 최종 후보를 정한 뒤 **hold-out Test는 딱 한 번만** 사용합니다.

> 탐색 시간은 각 탐색 도구의 내부 동작과 실행 환경에 따라 포함되는 작업이 조금 다를 수 있으므로, 이번 실습에서는 절대적인 속도 순위가 아니라 **대략적인 탐색 비용 비교**로 해석합니다.

## 요구사항

1. Grid / Random / Optuna 결과를 다음 컬럼으로 하나의 DataFrame에 정리합니다.
   - `search_method`
   - `best_params`
   - `cv_mean`
   - `cv_std`
   - `search_seconds`
2. 세 후보를 비교합니다.
3. CV 평균을 우선 기준으로 최종 후보를 선택하고, 평균이 거의 비슷하다면 CV 표준편차와 탐색 비용도 함께 고려하여 선택 이유를 설명합니다.
4. 튜닝하지 않은 기준 모델(`RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)`)과 3번에서 선택한 튜닝 후보를 비교하세요. 필수 실습의 `cv=5`와 동일한 `KFold(n_splits=5, shuffle=False)` 분할을 두 모델에 공통 적용하고, `cross_validate(..., scoring="r2", return_train_score=True)`로 평가하세요.
   - Fold별 Train R²와 검증 R²를 `fold_comparison`으로 정리합니다.
   - Train R² 평균, 검증 R² 평균·표준편차, `Train 평균 - 검증 평균`을 `tuning_comparison`으로 정리합니다.
   - `튜닝 후 CV 평균 R² - 기준 모델 CV 평균 R²`를 계산합니다. R² 차이값으로 보고하며, 백분율 개선율로 바꾸지 않습니다.
   - 평균·변동·Train–검증 차이로 개선 여부를 설명합니다. 튜닝이 개선되지 않았다면 기준 모델 유지도 가능합니다. Test 확인 전에 최종 모델과 선택 이유를 확정합니다.
   - 최종 선택 모델을 Train 전체에 다시 학습합니다.
5. hold-out Test에서 `.score()`로 최종 R²를 딱 한 번 확인합니다.
6. 최종 선택 모델의 CV 평균과 hold-out Test R²의 차이를 계산하고, 과적합 가능성과 일반화 판단의 한계를 설명합니다. Test 결과를 본 뒤 다시 튜닝하거나 후보를 교체하지 않습니다.
7. 선택한 모델을 `model_artifacts/best_model.pkl`로 저장합니다.
8. 다음 정보를 `model_artifacts/model_meta.json`에 저장합니다.
   - 모델 이름
   - 저장 시각
   - 선택한 탐색 방법
   - 최적 파라미터
   - CV 평균
   - CV 표준편차
   - 최종 Test R²
   - 기준 모델 설정, 튜닝 후보 설정·탐색 방법
   - 튜닝 전후 CV 비교 결과와 CV 평균 R² 차이
   - 선택 모델 CV 평균과 Test R²의 차이

### 확인 포인트

- Test 데이터를 탐색 중간에 사용하지 않았는가?
- CV 평균을 우선으로 보고, 필요할 때 표준편차와 탐색 시간도 함께 고려했는가?
- 최종 모델을 정한 뒤 Test를 한 번만 확인했는가?
- 모델뿐 아니라 메타데이터도 저장했는가?

- 튜닝 전후를 동일한 Fold에서 비교하고 개선 여부를 수치로 설명했는가?
- CV의 `test_score`는 Fold 내부 검증 점수이며 hold-out Test 점수가 아님을 구분했는가?
- 작은 표준편차나 CV/Test 점수의 일치만으로 일반화가 보장된다고 단정하지 않았는가?


In [ ]:
# TODO
# 세 탐색 후보를 비교하고 기준 모델과 튜닝 후보의 동일 Fold CV 결과를 정리하세요.
# 튜닝 전후 R² 차이를 계산해 최종 선택을 확정한 뒤
# 선택 모델만 hold-out Test에서 한 번 평가하고 모델·비교 메타데이터를 저장하세요.
from sklearn.model_selection import KFold, cross_validate

search_summary = pd.DataFrame([
    {
        "search_method": "GridSearchCV",
        "best_params": grid.best_params_,
        "cv_mean": grid_cv_mean,
        "cv_std": grid_cv_std,
        "search_seconds": grid_seconds,
    },
    {
        "search_method": "RandomizedSearchCV",
        "best_params": rand.best_params_,
        "cv_mean": rand_cv_mean,
        "cv_std": rand_cv_std,
        "search_seconds": rand_seconds,
    },
    {
        "search_method": "Optuna",
        "best_params": study.best_params,
        "cv_mean": optuna_cv_mean,
        "cv_std": optuna_cv_std,
        "search_seconds": optuna_seconds,
    },
])
display(search_summary)

# CV 평균 우선 선택 (동점이면 std·시간 고려)
best_row = search_summary.sort_values(
    ["cv_mean", "cv_std", "search_seconds"],
    ascending=[False, True, True],
).iloc[0]
print("선택된 탐색:", best_row["search_method"], best_row["best_params"])

baseline = RandomForestRegressor(
    n_estimators=100, max_depth=None, random_state=42, n_jobs=-1
)
tuned = RandomForestRegressor(
    **best_row["best_params"], random_state=42, n_jobs=-1
)

cv = KFold(n_splits=5, shuffle=False)
base_cv = cross_validate(
    baseline, X_train, y_train, cv=cv, scoring="r2", return_train_score=True
)
tuned_cv = cross_validate(
    tuned, X_train, y_train, cv=cv, scoring="r2", return_train_score=True
)

fold_comparison = pd.DataFrame({
    "fold": range(1, 6),
    "baseline_train_r2": base_cv["train_score"],
    "baseline_valid_r2": base_cv["test_score"],
    "tuned_train_r2": tuned_cv["train_score"],
    "tuned_valid_r2": tuned_cv["test_score"],
})
display(fold_comparison)

tuning_comparison = pd.DataFrame([
    {
        "model": "baseline",
        "train_r2_mean": base_cv["train_score"].mean(),
        "valid_r2_mean": base_cv["test_score"].mean(),
        "valid_r2_std": base_cv["test_score"].std(),
        "train_minus_valid": base_cv["train_score"].mean() - base_cv["test_score"].mean(),
    },
    {
        "model": "tuned",
        "train_r2_mean": tuned_cv["train_score"].mean(),
        "valid_r2_mean": tuned_cv["test_score"].mean(),
        "valid_r2_std": tuned_cv["test_score"].std(),
        "train_minus_valid": tuned_cv["train_score"].mean() - tuned_cv["test_score"].mean(),
    },
])
display(tuning_comparison)

r2_gain = tuned_cv["test_score"].mean() - base_cv["test_score"].mean()
print("튜닝 후 - 기준 CV 평균 R²:", r2_gain)

# 최종 모델 확정: 개선되면 tuned, 아니면 baseline 유지 가능
if r2_gain > 0:
    final_model = tuned
    final_method = best_row["search_method"]
    final_params = best_row["best_params"]
    final_cv_mean = tuned_cv["test_score"].mean()
    final_cv_std = tuned_cv["test_score"].std()
else:
    final_model = baseline
    final_method = "baseline"
    final_params = {"n_estimators": 100, "max_depth": None}
    final_cv_mean = base_cv["test_score"].mean()
    final_cv_std = base_cv["test_score"].std()

final_model.fit(X_train, y_train)
test_r2 = final_model.score(X_test, y_test)
print("Hold-out Test R²:", test_r2)
print("CV mean - Test R²:", final_cv_mean - test_r2)

os.makedirs("model_artifacts", exist_ok=True)
joblib.dump(final_model, "model_artifacts/best_model.pkl")

meta = {
    "model_name": "RandomForestRegressor",
    "saved_at": datetime.now().isoformat(timespec="seconds"),
    "search_method": str(final_method),
    "best_params": final_params if isinstance(final_params, dict) else dict(final_params),
    "cv_mean": float(final_cv_mean),
    "cv_std": float(final_cv_std),
    "test_r2": float(test_r2),
    "baseline_setting": {"n_estimators": 100, "max_depth": None},
    "tuned_candidate": {
        "search_method": str(best_row["search_method"]),
        "best_params": best_row["best_params"],
    },
    "cv_r2_gain_tuned_minus_baseline": float(r2_gain),
    "cv_mean_minus_test_r2": float(final_cv_mean - test_r2),
}
# best_params may contain numpy types
def _to_py(o):
    if isinstance(o, dict):
        return {k: _to_py(v) for k, v in o.items()}
    if hasattr(o, "item"):
        try:
            return o.item()
        except Exception:
            return o
    return o

with open("model_artifacts/model_meta.json", "w", encoding="utf-8") as f:
    json.dump(_to_py(meta), f, ensure_ascii=False, indent=2)

print("저장 완료: model_artifacts/best_model.pkl, model_meta.json")


### Q4. CV 평균이 아주 조금 높은 모델의 탐색 시간이 몇 배 더 길다면 무조건 그 모델을 선택해야 하나요?

**답변:**  
아니요. 성능 차이가 미미하면 표준편차·탐색 비용·운영 제약을 함께 보고 실용적으로 선택합니다.


### Q5. hold-out Test를 탐색 마지막에 한 번만 사용해야 하는 이유는 무엇인가요?

**답변:**  
탐색 중간에 반복 사용하면 Test에 맞춰 튜닝하는 누수가 생겨 일반화 추정이 낙관적으로 왜곡됩니다.

### Q6. 튜닝 전후 CV 평균 R²는 얼마나 달라졌나요? Fold별 변동과 Train–검증 점수 차이를 근거로 개선 여부를 설명하세요.

**답변:**
`tuning_comparison`과 `r2_gain` 출력값을 근거로 설명합니다. 평균이 올라도 Fold 표준편차·Train–Valid gap이 커지면 개선을 단정하지 않습니다.

### Q7. CV 결과와 최종 Test 결과를 어떻게 해석했나요? 과적합 가능성과 일반화 판단의 한계를 설명하세요.

**답변:**
CV 평균과 Test R² 차이(`cv_mean - test_r2`)가 크면 과적합·분포 차이 가능성을 의심합니다. 한 번의 Test만으로 일반화를 보장한다고 단정하지 않습니다.


---

# 실습 마무리

1. GridSearchCV와 RandomizedSearchCV의 가장 큰 탐색 방식 차이는 무엇인가요?
2. RandomizedSearchCV의 `n_iter`는 무엇을 의미하나요?
3. Optuna에서 `study`와 `trial`은 각각 무엇인가요?
4. 최종 모델을 고를 때 CV 평균 외에 무엇을 함께 봐야 하나요?
5. hold-out Test는 언제 사용해야 하나요?
6. 최종 튜닝 모델과 메타데이터를 저장하는 이유는 무엇인가요?